# 예제 05. pandas 오류와 경고
빅데이터프로그래밍 · 3주차

아래 셀들은 **일부러 오류와 경고가 나도록** 만들어져 있습니다.
전처리에서 실제로 만나는 것들입니다.


In [ ]:
csv = """name,gender,department,study_hours,attendance,midterm,final
김통계,남,통계학과,12.5,95%,88,92
이확률,여,통계학과,8.0,88%,92,85
박회귀,남,컴퓨터공학과,,72%,79,68
최추정,여,통계학과,15.0,100%,95,98
정검정,남,경제학과,4.5,61%,61,
한분산,여,컴퓨터공학과,10.0,90%,84,88
오평균,남,경제학과,6.5,,70,74
서표본,여,통계학과,13.0,97%,100,96
남표준,남,컴퓨터공학과,9.5,85%,66,71
윤편차,여,경제학과,,80%,82,79
"""

with open("students.csv", "w", encoding="utf-8") as f:
    f.write(csv)

print("students.csv 생성 완료")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("students.csv")
df.head(3)


## 1. KeyError — 없는 열 이름


In [ ]:
df["Midterm"]


In [ ]:
# 해결: 열 이름을 확인한다. 대소문자와 공백까지 정확히 같아야 합니다
print(df.columns.tolist())
print(df["midterm"].head(3))


## 2. TypeError — object 열로 계산했다


In [ ]:
df["attendance"].mean()


In [ ]:
# 해결: 숫자로 바꾼 뒤 계산
att = df["attendance"].str.replace("%", "", regex=False).astype(float)
print(att.mean())


## 3. ValueError — 숫자로 바꿀 수 없는 값이 섞여 있다


In [ ]:
df["attendance"].astype(float)


In [ ]:
# 해결 1: 문자를 먼저 제거
# 해결 2: 변환 실패를 NaN 으로 두고 나중에 처리
att = pd.to_numeric(df["attendance"].str.replace("%", "", regex=False),
                    errors="coerce")
print(att)
print("변환 실패(NaN):", att.isnull().sum())


## 4. NaN 은 조용히 퍼집니다 — 오류가 나지 않아 더 위험합니다


In [ ]:
print("final 합계 :", df["final"].sum())        # NaN 을 건너뜁니다
print("final 평균 :", df["final"].mean())
print()
print("NaN + 10 =", df["final"].iloc[4] + 10)     # NaN 이 그대로 전파


In [ ]:
# 해결: 학습 직전에 반드시 확인
print("결측 개수:\n", df.isnull().sum())


## 5. SettingWithCopyWarning — 사본을 고쳤을 수 있다


In [ ]:
subset = df[df["midterm"] >= 80]
subset["midterm"] = 100          # 경고 발생
subset.head(3)


In [ ]:
# 해결: copy() 를 명시하거나 loc 으로 원본에 직접 쓴다
subset = df[df["midterm"] >= 80].copy()
subset["midterm"] = 100

df.loc[df["midterm"] >= 80, "midterm"] = 100
print(df["midterm"].tolist())


## 6. Tensor 변환에서 나는 오류


In [ ]:
import torch

# object 열이 남아 있으면 변환이 실패합니다
raw = pd.read_csv("students.csv")
torch.from_numpy(raw.to_numpy())


In [ ]:
# 해결: 숫자 열만, float32 로
num = raw.select_dtypes(include="number").fillna(0).to_numpy(dtype="float32")
t = torch.from_numpy(num)
print(t.shape, t.dtype)


## 직접 해보기
아래 셀에는 문제가 두 개 있습니다. 오류 메시지와 경고를 읽고 고치세요.


In [ ]:
data = pd.read_csv("students.csv")
high = data[data["Final"] > 80]
high["final"] = 100
print(high)
